# Data Explorer: scores, groupings, preprocessing

**Purpose:** Exploratory analysis of the conformity dataset. Loads the SQLite DB (`events`, `cleaned`, `scores`), summarizes preprocessing coverage, visualizes score distributions across metadata dimensions (repo, event type, actor, time), and shows samples of comments.

**Working directory:** The first code cell adds the repo root to `sys.path`, so you can run with the kernel's cwd as either the repository root or the `notebooks/` folder. The database path is still `data/raw/events.db` under the repo.

In [ ]:
import sys
from pathlib import Path

# Add repo root to sys.path so notebooks.lib can be imported
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "project_config.py").is_file():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

In [ ]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from notebooks.lib import connect, summarize_by, bucket_top_n_series, plot_group_means, plot_heatmap

%matplotlib inline

# Judge model to analyze (must match scores.model_name)
MODEL_NAME = "gpt-5.4-mini"

RANDOM_SEED = 42
CLEAN_SAMPLE_N = 12
ACTOR_TOP_N = 15
ACTOR_MIN_N = 5

SCORE_COLS = ["fun_score", "nsi_score", "insi_score", "isi_score"]

## Available judge models

If the main query returns no rows, pick a `model_name` from this list.

In [ ]:
with connect() as conn:
    models = pd.read_sql("SELECT model_name, COUNT(*) AS n FROM scores GROUP BY model_name ORDER BY n DESC", conn)
models

## Load analysis frame (cleaned + events + scores)

`author_association` uses the same fallback order as `preprocessing.workflow._get_author_association` (comment → review → pull_request → issue).

In [ ]:
SQL_ANALYSIS = """
SELECT
  c.id AS comment_id,
  c.cleaned_text,
  json_extract(e.event_data, '$.type') AS event_type,
  json_extract(e.event_data, '$.repo.name') AS repo,
  json_extract(e.event_data, '$.created_at') AS created_at,
  date(json_extract(e.event_data, '$.created_at')) AS event_date,
  COALESCE(
    json_extract(e.event_data, '$.actor.login'),
    json_extract(e.event_data, '$.actor.display_login'),
    ''
  ) AS actor_login,
  COALESCE(
    json_extract(e.event_data, '$.payload.comment.author_association'),
    json_extract(e.event_data, '$.payload.review.author_association'),
    json_extract(e.event_data, '$.payload.pull_request.author_association'),
    json_extract(e.event_data, '$.payload.issue.author_association'),
    ''
  ) AS author_association,
  s.fun_score,
  s.nsi_score,
  s.insi_score,
  s.isi_score,
  s.model_name
FROM cleaned c
INNER JOIN events e ON e.id = c.id
INNER JOIN scores s ON s.comment_id = c.id
WHERE s.model_name = ?
"""

with connect() as conn:
    df = pd.read_sql(SQL_ANALYSIS, conn, params=[MODEL_NAME])

for col in SCORE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["actor_login"] = df["actor_login"].fillna("").astype(str)
df["author_association"] = df["author_association"].fillna("").astype(str)
df["event_type"] = df["event_type"].fillna("").astype(str)
df["repo"] = df["repo"].fillna("").astype(str)

print(f"Loaded {len(df)} rows for model {MODEL_NAME}")
df.head(3)

## Preprocessing and coverage

- **`events`**: every raw GitHub event ingested by `dataset.py`.
- **`cleaned`**: rows that survived `preprocess.py` (text extraction, bot filter, stripping, token count ≫ 2). The current `default_workflow()` in `preprocessing/workflow.py` has **`filter_trivial` commented out**, so trivial comments are not dropped by that step.
- **`scores`**: LLM judge output for a subset of cleaned comments.

The metrics table below uses **`count`** (integer row counts) and **`percent`** (coverage as 0–100, two decimal places; not applicable for pure count rows).

**Limitation:** Per-step drop counts (bots, missing text, too few tokens, duplicate event ids, etc.) are **not stored in SQLite**. Only aggregate `read / duplicates / kept` appears in logs when you run `preprocess.py`. A full funnel breakdown needs instrumented preprocessing or saved logs—not derivable from the DB alone beyond overall `|events|` vs `|cleaned|`.

In [ ]:
SQL_CLEANED_SAMPLE = """
SELECT
  c.id AS comment_id,
  json_extract(e.event_data, '$.type') AS event_type,
  json_extract(e.event_data, '$.repo.name') AS repo,
  date(json_extract(e.event_data, '$.created_at')) AS event_date,
  COALESCE(
    json_extract(e.event_data, '$.actor.login'),
    json_extract(e.event_data, '$.actor.display_login'),
    ''
  ) AS actor_login,
  substr(c.cleaned_text, 1, 200) AS cleaned_text_preview
FROM cleaned c
INNER JOIN events e ON e.id = c.id
"""

with connect() as conn:
    n_events = int(pd.read_sql("SELECT COUNT(*) AS n FROM events", conn)["n"].iloc[0])
    n_cleaned = int(pd.read_sql("SELECT COUNT(*) AS n FROM cleaned", conn)["n"].iloc[0])
    n_scores_total = int(pd.read_sql("SELECT COUNT(*) AS n FROM scores", conn)["n"].iloc[0])
    n_scored_model = int(
        pd.read_sql(
            "SELECT COUNT(*) AS n FROM scores WHERE model_name = ?",
            conn,
            params=[MODEL_NAME],
        )["n"].iloc[0]
    )
    n_scored_distinct = int(
        pd.read_sql(
            "SELECT COUNT(DISTINCT comment_id) AS n FROM scores WHERE model_name = ?",
            conn,
            params=[MODEL_NAME],
        )["n"].iloc[0]
    )
    cleaned_sample = pd.read_sql(SQL_CLEANED_SAMPLE, conn)

pct_cleaned_vs_events = round(100.0 * n_cleaned / n_events, 2) if n_events else float("nan")
pct_scored_vs_cleaned = round(100.0 * n_scored_distinct / n_cleaned, 2) if n_cleaned else float("nan")

preprocess_metrics = pd.DataFrame(
    [
        {"metric": "events (raw rows)", "count": n_events, "percent": pd.NA},
        {"metric": "cleaned (after preprocess)", "count": n_cleaned, "percent": pd.NA},
        {"metric": f"scores rows (model={MODEL_NAME})", "count": n_scored_model, "percent": pd.NA},
        {"metric": f"distinct scored comments (model={MODEL_NAME})", "count": n_scored_distinct, "percent": pd.NA},
        {"metric": "scores rows (all models)", "count": n_scores_total, "percent": pd.NA},
        {"metric": "coverage cleaned / events (%)", "count": pd.NA, "percent": pct_cleaned_vs_events},
        {"metric": "coverage scored / cleaned, distinct (%)", "count": pd.NA, "percent": pct_scored_vs_cleaned},
    ]
)
preprocess_metrics["count"] = preprocess_metrics["count"].astype("Int64")
preprocess_metrics["percent"] = preprocess_metrics["percent"].astype("Float64")
preprocess_metrics

In [ ]:
sample_cleaned = cleaned_sample.sample(
    n=min(CLEAN_SAMPLE_N, len(cleaned_sample)),
    random_state=RANDOM_SEED,
).sort_values(["repo", "event_date"])
sample_cleaned

### Actor "role" in this dataset

- **`author_association`** (above) is GitHub's label for the author's relationship to the repo on that object (e.g. `MEMBER`, `OWNER`). It is not org job title.
- Richer roles (maintainer lists, teams) require the GitHub API or an external mapping merged by `login`.

## Mean scores by dimension

Overall means for the selected model (same row set as `df`).

In [ ]:
if len(df) == 0:
    print("No rows: check MODEL_NAME against the models table above.")
else:
    display(df[SCORE_COLS].mean().to_frame("mean").T)
    display(df[SCORE_COLS].describe().T)

## Grouped means and counts

Break down scores by single and paired dimensions. Bars show mean scores; **n** is count per group.

In [ ]:
if len(df):
    # Single dimensions
    for col in ["event_type", "repo", "author_association"]:
        summ = summarize_by(df, col)
        print(f"\n{col} — {len(summ)} groups")
        display(summ.head(20))
        plot_group_means(summ.sort_values(col), col, f"Mean scores by {col}")

    # By week
    by_week = df.copy()
    by_week["event_week"] = by_week["event_date"].dt.to_period("W").astype(str)
    summ_w = summarize_by(by_week, "event_week")
    print(f"\nevent_week — {len(summ_w)} weeks")
    display(summ_w.head(20))
    plot_group_means(summ_w.sort_values("event_week"), "event_week", "Mean scores by week")

    # By actor (top N)
    df_actor = df.copy()
    df_actor["actor_bucket"] = bucket_top_n_series(df_actor["actor_login"], ACTOR_TOP_N)
    summ_a = summarize_by(df_actor, "actor_bucket", min_n=ACTOR_MIN_N)
    print(f"\nactor_login (top {ACTOR_TOP_N} + Other, min n={ACTOR_MIN_N}) — {len(summ_a)} groups")
    display(summ_a.head(25))
    plot_group_means(summ_a.sort_values("n", ascending=False), "actor_bucket", f"Mean scores by actor (top {ACTOR_TOP_N} + Other)")

## Two-way groupings (heatmap)

Example: repo × event_type for **NSI**; change `value` to `insi_score`, `isi_score`, or `fun_score` as needed.

In [ ]:
if len(df):
    g_rt = ["repo", "event_type"]
    means_rt = df.groupby(g_rt, dropna=False)[SCORE_COLS].mean().reset_index()
    counts_rt = df.groupby(g_rt, dropna=False).size().reset_index(name="n")
    tw = means_rt.merge(counts_rt, on=g_rt)
    tw = tw[tw["n"] >= ACTOR_MIN_N]
    display(tw.sort_values("n", ascending=False).head(30))
    plot_heatmap(tw, row="repo", col="event_type", value="nsi_score", title="Mean NSI by repo × event_type")

    g_ra = ["repo", "author_association"]
    means_ra = df.groupby(g_ra, dropna=False)[SCORE_COLS].mean().reset_index()
    counts_ra = df.groupby(g_ra, dropna=False).size().reset_index(name="n")
    tw2 = means_ra.merge(counts_ra, on=g_ra)
    tw2 = tw2[tw2["n"] >= ACTOR_MIN_N]
    display(tw2.sort_values("n", ascending=False).head(30))
    plot_heatmap(
        tw2,
        row="repo",
        col="author_association",
        value="isi_score",
        title="Mean ISI by repo × author_association",
    )